# 使用 PROC GAM 建模零售业非线性需求曲线

## 摘要

本笔记本使用 **PROC GAM** 将周度杂货零售单位销量建模为货架价格、门店温度（作为季节性
代理变量）与促销支出的平滑非线性函数，并加入一个参数化的促销标志效应。广义可加模型让
品类经理能够还原线性回归会遗漏的真实曲线型价格弹性与季节性需求形状，从而支持更精准的
定价与促销决策。

## 数据来源

| 数据集 | 行数 | 粒度 | 关键变量 | 说明 |
|---------|------|-------|---------------|-------------|
| `store_sales` | 100 | 周 | `Week`, `Price`, `Temperature`, `Promotion`, `PromoSpend`, `Units` | 某旗舰杂货门店连续100周（近两个季节周期）的合成周度销售点（POS）数据。通过
 `call streaminit(20250531)` 与 `rand()` 在程序内生成。周度单位销量遵循一个泊松需求
过程，其均值由指数型价格响应曲线、在约72华氏度附近达到峰值的二次温度/季节性效应，以及
凹形（平方根）促销支出提升效应共同驱动，此外还叠加一个离散的促销周标志。 |

# 使用 PROC GAM 建模零售业非线性需求曲线

零售需求对价格、天气或促销支出的响应很少是直线型的。主打商品的小幅降价可能几乎不影响
销量，而跨过某个心理价位却可能引发销量的急剧跃升；天气驱动的需求在一个舒适的中间区间
达到峰值，并在两端回落；促销带来的提升则随着支出增加而呈现边际递减。

**PROC GAM**（广义可加模型）为每个驱动因素拟合各自的平滑样条项，因此是数据本身——而非
僵化的线性假设——决定了每条需求曲线的形状。这里我们为某单一旗舰杂货门店连续100周的
周度单位销量建模，将参数化的促销标志与价格、温度、促销支出的平滑样条相结合，采用泊松
响应。

## 第1步 —— 生成合成周度销售序列

我们模拟某旗舰门店连续100周（近两个季节周期）的销售点数据。数据生成过程被刻意设计为
非线性，以便我们能够确认 GAM 能够还原真实的曲线形状：

- **Price（价格）** 通过指数型响应曲线（`exp(-1.1 * Price)`）驱动销量，因此价格越低，
  需求上升越陡峭。
- **Temperature（温度）** 作为季节性代理变量，在约72华氏度附近呈现二次曲线峰值，反映
  舒适度驱动的客流量。
- **PromoSpend（促销支出）** 带来凹形的平方根提升效应（边际收益递减）。
- 离散的 **Promotion（促销）** 标志在促销周上叠加一个参数化的阶跃效应。

周度 `Units`（单位销量）取自泊松分布，与单位销量的计数性质相匹配。

In [1]:
数据 store_sales;
   调用 streaminit(20250531);
   长度 Promotion $3;
   循环 Week = 1 到 100;
      BasePrice = 3.20 + 0.30 * rand("uniform");
      Discount  = 0.40 * rand("uniform");
      Price     = round(BasePrice - Discount, 0.01);
      如果 rand("uniform") < 0.28 那么 循环;
         Promotion  = "Yes";
         PromoSpend = round(200 + 600 * rand("uniform"), 1);
      结束;
      否则 循环;
         Promotion  = "No";
         PromoSpend = 0;
      结束;
      Temperature = 55 + 25 * sin((Week - 12) / 52 * 2 * 3.14159265)
                    + 4 * rand("normal");
      priceEffect = 180 * EXP(-1.1 * Price);
      tempEffect  = -0.012 * (Temperature - 72) ** 2;
      promoEffect = 0.085 * sqrt(PromoSpend);
      logMu = 3.0 + LOG(priceEffect) + tempEffect + promoEffect;
      Units = rand("poisson", EXP(logMu) / 12);
      输出;
   结束;
运行;



NOTE: DATA store_sales


NOTE: Wrote store_sales (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## 第2步 —— 剖析模拟数据

一次简单的 `PROC MEANS` 确认各变量在建模前落在合理的零售区间内：单位销量为非负整数，
价格集中在几美元左右，温度经历完整的季节周期，促销支出在非促销周为零。

In [2]:
过程 均值 数据=store_sales n mean MIN MAX maxdec=2;
   变量 Units Price Temperature PromoSpend;
   标签 Units="销量" Price="价格" Temperature="温度" PromoSpend="促销支出";
运行;


                                                  The MEANS Procedure

 Variable     Label                N           Mean     Minimum     Maximum
 --------------------------------------------------------------------------
 Units        销量                 100           7.03        0.00      103.00
 Price        价格                 100           3.15        2.81        3.48
 Temperature  温度                 100          55.50       22.72       83.49
 PromoSpend   促销支出               100         128.76        0.00      779.00
 --------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 第3步 —— 拟合完整的加法型需求模型

核心模型组合了以下部分：

- `param(Promotion)` —— 在 `CLASS` 语句中声明的、针对促销周指示变量的参数化（线性）
  效应。
- `spline(Price, df=4)` —— 捕捉曲线型价格响应的三次平滑样条。
- `spline(Temperature, df=4)` —— 平滑的季节性曲线。
- `spline(PromoSpend, df=3)` —— 边际递减的促销提升效应。

由于单位销量是计数型数据，我们指定 `dist=poisson`（对数联系函数）。`method=gcv`
让广义交叉验证在无需显式指定自由度的情况下引导各分量的平滑程度。`OUTPUT` 语句将逐
观测值的预测值与残差写入 `gam_fit`。

该过程报告的 **偏差为 43.59**，相对于 **零偏差 2041.12**——四个平滑加参数化驱动因素
几乎解释了常数模型所遗漏的全部变异——以及 **AIC 为 254.61**。参数化的
`PROMOTIONYES` 估计值为正（对数尺度上 +0.41），确认了促销带来的销量提升，而价格样条
呈现出较强的负线性趋势（−1.71），这正是需求随价格下降的典型特征。

In [3]:
过程 gam 数据=store_sales PLOTS=components(CLM commonaxes);
   分类 Promotion;
   模型 Units = PARAM(Promotion)
                 SPLINE(Price, df=4)
                 SPLINE(Temperature, df=4)
                 SPLINE(PromoSpend, df=3) / DIST=poisson METHOD=gcv;
   输出 out=gam_fit predicted residual;
   标签 Units="销量" Promotion="促销" Price="价格"
         Temperature="温度" PromoSpend="促销支出";
运行;



                                                   The GAM Procedure                                                    

Model Information
Response Variable     销量
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        43.592828
Null Deviance   2041.115451
AIC             254.611076

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                5.228805       0.000000       0.000000       0.000000
PROMOTIONYES               0.406972       0.000000       0.000000       0.000000
S(PRICE, DF = 4)          -1.705326       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.031495       0.000000       0.000000       0.000000
S(PROMOSPEND, DF = 3)       0.002307       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(价格)                       4.0000         4.000


NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## 第4步 —— 聚焦价格与季节性的模型

为了得到更精简的定价评审模型，我们仅保留与决策最相关的两个平滑驱动因素——价格与温度
——重新拟合模型，并为价格提供额外的灵活性（`df=5`）以捕捉心理价位附近可能出现的拐点。
促销标志则作为参数化控制项保留。

去掉促销支出后，**偏差上升到 62.80**，**AIC 上升到 269.82**，均高于完整模型的
43.59 与 254.61。参数化的 `PROMOTIONYES` 项在此处也吸收了更多促销信号
（+1.73，相比完整模型的 +0.41），因为不再有促销支出样条来承担这部分效应。价格样条
仍保持负向趋势（−1.51），说明核心的需求故事在不同设定下保持稳定。

In [4]:
过程 gam 数据=store_sales PLOTS=components(CLM);
   分类 Promotion;
   模型 Units = PARAM(Promotion)
                 SPLINE(Price, df=5)
                 SPLINE(Temperature, df=4) / DIST=poisson;
   标签 Units="销量" Promotion="促销" Price="价格"
         Temperature="温度";
运行;



                                                   The GAM Procedure                                                    

Model Information
Response Variable     销量
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        62.803733
Null Deviance   2041.115451
AIC             269.821548

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                4.915205       0.000000       0.000000       0.000000
PROMOTIONYES               1.725573       0.000000       0.000000       0.000000
S(PRICE, DF = 5)          -1.511509       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.027370       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(价格)                       5.0000         5.0000
Spline(温度)                       4.0000         4.0000





NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## 结果解读

**回归模型分析**表报告了每个分量内的线性趋势以及参数化的促销效应。正的 `PROMOTIONYES`
系数（完整模型中为 +0.41，精简模型中为 +1.73）确认了预期中的促销销量提升，而价格样条
上的负线性趋势（−1.71 与 −1.51）体现了经典的价格下降型需求曲线。温度样条较小的正线性
项（+0.03）在预期之内：其真正的故事在于围绕72华氏度舒适峰值的曲率，单一线性系数无法
概括这一点。

**平滑模型分析**表报告了每个样条所消耗的自由度。价格与温度各自使用4个有效自由度
（精简模型中价格为5个），促销支出使用3个——均远高于直线所使用的单一自由度，这正是
线性回归会遗漏这些曲线关系的原因。

**拟合统计量**让你可以直接比较两种设定。完整的四驱动因素模型给出偏差43.59与AIC
254.61，优于精简模型的62.80与269.82；两项指标都支持完整模型，说明促销支出与促销
标志在价格与温度之外还提供了额外的解释力。相对于2041.12的零偏差，两个模型都捕捉到了
绝大部分的需求变异。

综合来看，这些表格为品类经理提供了一个量化的、数据驱动的需求故事：陡峭的价格响应可以
指导降价深度，季节性的温度窗口，以及边际递减的促销支出效应——远比单一的线性弹性估计
更为精准。（PROC GAM 也支持 `plots=components` 来绘制每个平滑项的部分预测曲线；上面
的数值分量表正是这些曲线的数据来源。）